2.0.1 Drive i putanje

Montira se Google Drive i definišu se glavne putanje projekta, posebno folderi za curated podatke i za rezultate. Na kraju se ispisuju putanje da odmah vidiš da li Colab gleda na pravi direktorijum.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, math, time
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Personal/Diplomski")
CURATED = ROOT / "Data" / "curated"
RUNS = ROOT / "Runs"
RUNS.mkdir(parents=True, exist_ok=True)

DATASETS = ["thyroid_recurrence", "lc25000", "sipakmed", "rm1000_lung_history"]

print("ROOT:", ROOT)
print("CURATED:", CURATED)
print("RUNS:", RUNS)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/Personal/Diplomski
CURATED: /content/drive/MyDrive/Personal/Diplomski/Data/curated
RUNS: /content/drive/MyDrive/Personal/Diplomski/Runs


2.0.1 Drive i putanje

Montira se Google Drive i definišu se glavne putanje projekta, posebno folderi za curated podatke i za rezultate. Na kraju se ispisuju putanje da odmah vidiš da li Colab gleda na pravi direktorijum.

In [2]:
import pandas as pd
import numpy as np

def load_meta(ds):
    p = CURATED / ds / "meta.json"
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def load_split(ds, split):
    p = CURATED / ds / f"{split}.csv"
    return pd.read_csv(p)

def infer_task(meta):
    t = meta.get("task_type")
    if t in ["image", "tabular"]:
        return t
    if "image" in json.dumps(meta).lower():
        return "image"
    return "tabular"

def label_col(meta, df):
    c = meta.get("label_col")
    if c and c in df.columns:
        return c
    for cand in ["label", "target", "y", "class", "diagnosis"]:
        if cand in df.columns:
            return cand
    for col in df.columns[::-1]:
        if df[col].dtype == "object":
            continue
    return df.columns[-1]

def path_col(meta, df):
    c = meta.get("path_col")
    if c and c in df.columns:
        return c
    for cand in ["path", "filepath", "image_path", "img_path", "file"]:
        if cand in df.columns:
            return cand
    return None

def check_image_paths(df, pcol, max_check=2000):
    if pcol is None:
        return {"checked": 0, "missing": 0, "missing_examples": []}
    n = min(len(df), max_check)
    idx = np.random.choice(len(df), size=n, replace=False) if len(df) else []
    miss = []
    for i in idx:
        p = str(df.iloc[i][pcol])
        if not os.path.exists(p):
            miss.append(p)
            if len(miss) >= 10:
                break
    return {"checked": n, "missing": len(miss), "missing_examples": miss}

def class_counts(df, ycol):
    vc = df[ycol].value_counts(dropna=False)
    return vc.to_dict()

def dup_stats(df, cols):
    if cols is None:
        return {"dups": 0}
    d = df.duplicated(subset=cols).sum()
    return {"dups": int(d)}

def sanity_report(ds):
    meta = load_meta(ds)
    splits = {}
    for sp in ["train", "val", "test"]:
        df = load_split(ds, sp)
        splits[sp] = df

    task = infer_task(meta)
    sample_df = splits["train"]
    ycol = label_col(meta, sample_df)
    pcol = path_col(meta, sample_df) if task == "image" else None

    out = {"dataset": ds, "task": task, "label_col": ycol, "path_col": pcol}
    for sp, df in splits.items():
        d = {"rows": int(len(df))}
        d.update(dup_stats(df, [pcol] if pcol else None))
        d["class_counts"] = class_counts(df, ycol)
        if task == "image":
            d["path_check"] = check_image_paths(df, pcol)
        out[sp] = d
    return out

reports = []
for ds in DATASETS:
    r = sanity_report(ds)
    reports.append(r)

pd.DataFrame([{
    "dataset": r["dataset"],
    "task": r["task"],
    "train": r["train"]["rows"],
    "val": r["val"]["rows"],
    "test": r["test"]["rows"],
    "label_col": r["label_col"],
    "path_col": r["path_col"]
} for r in reports])

,dataset,task,train,val,test,label_col,path_col
0,thyroid_recurrence,tabular,255,54,55,Recurred,None
1,lc25000,image,17500,3750,3750,label,path
2,sipakmed,image,2835,608,606,label,path
3,rm1000_lung_history,image,10500,2250,2250,label,path


2.0.3 Sanity report

Za svaki dataset se učitavaju train, val i test CSV fajlovi i pravi se kratak izveštaj o broju uzoraka, duplikatima i raspodeli klasa. Kod slikovnih datasetova dodatno proverava da li putanje do fajlova zaista postoje na disku.

In [3]:
def pretty_counts(d):
    items = sorted(list(d.items()), key=lambda x: str(x[0]))
    return ", ".join([f"{k}:{v}" for k,v in items])

for r in reports:
    print("\n==============================")
    print("DATASET:", r["dataset"], "| TASK:", r["task"])
    print("label_col:", r["label_col"], "| path_col:", r["path_col"])
    for sp in ["train", "val", "test"]:
        cc = pretty_counts(r[sp]["class_counts"])
        print(f"{sp:>5} rows={r[sp]['rows']}, dups={r[sp]['dups']}, classes=({cc})")
        if r["task"] == "image":
            pc = r[sp]["path_check"]
            print("      path_check:", pc["checked"], "checked,", pc["missing"], "missing,", pc["missing_examples"])


DATASET: thyroid_recurrence | TASK: tabular
label_col: Recurred | path_col: None
train rows=255, dups=0, classes=(No:179, Yes:76)
  val rows=54, dups=0, classes=(No:38, Yes:16)
 test rows=55, dups=0, classes=(No:39, Yes:16)

DATASET: lc25000 | TASK: image
label_col: label | path_col: path
train rows=17500, dups=0, classes=(colon_image_sets:7000, lung_image_sets:10500)
      path_check: 2000 checked, 0 missing, []
  val rows=3750, dups=0, classes=(colon_image_sets:1500, lung_image_sets:2250)
      path_check: 2000 checked, 0 missing, []
 test rows=3750, dups=0, classes=(colon_image_sets:1500, lung_image_sets:2250)
      path_check: 2000 checked, 0 missing, []

DATASET: sipakmed | TASK: image
label_col: label | path_col: path
train rows=2835, dups=0, classes=(im_Dyskeratotic:569, im_Koilocytotic:578, im_Metaplastic:555, im_Parabasal:551, im_Superficial-Intermediate:582)
      path_check: 2000 checked, 0 missing, []
  val rows=608, dups=0, classes=(im_Dyskeratotic:122, im_Koilocytotic:12

2.0.4 Prikaz uzoraka slika

Nasumično se bira nekoliko slika iz trening skupa i prikazuju se u mreži zajedno sa labelama. Ovo je vizuelna potvrda da su putanje tačne i da SipakMed CROPPED varijanta izgleda kako treba.

In [4]:
import matplotlib.pyplot as plt
from PIL import Image

def show_samples(ds, n=12, seed=42):
    meta = load_meta(ds)
    df = load_split(ds, "train")
    task = infer_task(meta)
    if task != "image":
        print(ds, "nije image dataset.")
        return
    ycol = label_col(meta, df)
    pcol = path_col(meta, df)

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(df), size=min(n, len(df)), replace=False)

    cols = 4
    rows = math.ceil(len(idx)/cols)
    plt.figure(figsize=(cols*4, rows*4))
    for k, i in enumerate(idx, start=1):
        p = str(df.iloc[i][pcol])
        y = str(df.iloc[i][ycol])
        try:
            im = Image.open(p).convert("RGB")
        except:
            im = Image.new("RGB", (224,224))
            y = y + " (LOAD_FAIL)"
        plt.subplot(rows, cols, k)
        plt.imshow(im)
        plt.axis("off")
        plt.title(y)
    plt.suptitle(ds)
    plt.show()

for ds in ["lc25000", "sipakmed", "rm1000_lung_history"]:
    show_samples(ds, n=12, seed=42)

Output hidden; open in https://colab.research.google.com to view.

Baseline 1: Tabular (thyroid) Logistic Regression + RandomForest

2.1.1 Tabular baseline za thyroid

Trenira se jednostavan model za tabularne podatke da dobiješ početnu referencu performansi pre naprednijih metoda. Evaluacija se radi na val i test splitu uz accuracy i macro F1 da poređenje kasnije bude fer.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def tabular_xy(ds, split):
    meta = load_meta(ds)
    df = load_split(ds, split)
    ycol = label_col(meta, df)
    y = df[ycol]
    X = df.drop(columns=[ycol])
    return X, y, ycol

def eval_clf(name, clf, X_train, y_train, X_val, y_val, X_test, y_test):
    clf.fit(X_train, y_train)
    def metrics(X, y):
        p = clf.predict(X)
        return {
            "acc": float(accuracy_score(y, p)),
            "f1_macro": float(f1_score(y, p, average="macro"))
        }
    out = {
        "model": name,
        "val": metrics(X_val, y_val),
        "test": metrics(X_test, y_test),
        "report_test": classification_report(y_test, clf.predict(X_test), output_dict=True)
    }
    return out

ds = "thyroid_recurrence"
X_train, y_train, _ = tabular_xy(ds, "train")
X_val, y_val, _ = tabular_xy(ds, "val")
X_test, y_test, _ = tabular_xy(ds, "test")

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

lr = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, n_jobs=None))])
rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1))])

res_lr = eval_clf("LogReg", lr, X_train, y_train, X_val, y_val, X_test, y_test)
res_rf = eval_clf("RandomForest", rf, X_train, y_train, X_val, y_val, X_test, y_test)

pd.DataFrame([
    {"model": res_lr["model"], "val_acc": res_lr["val"]["acc"], "val_f1": res_lr["val"]["f1_macro"], "test_acc": res_lr["test"]["acc"], "test_f1": res_lr["test"]["f1_macro"]},
    {"model": res_rf["model"], "val_acc": res_rf["val"]["acc"], "val_f1": res_rf["val"]["f1_macro"], "test_acc": res_rf["test"]["acc"], "test_f1": res_rf["test"]["f1_macro"]},
])

,model,val_acc,val_f1,test_acc,test_f1
0,LogReg,0.925926,0.903571,0.945455,0.932626
1,RandomForest,0.962963,0.953846,0.981818,0.977542


Baseline 2: Image (transfer learning ResNet18)

2.2.1 Torch setup

Učitava se PyTorch i bira se uređaj za računanje, GPU ako postoji ili CPU ako ne. Time sve naredne operacije oko treninga idu na pravilnu metu.

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

2.2.2 Dataset i DataLoader iz CSV-a

Pravi se PyTorch dataset koji čita putanju slike i labelu iz CSV fajla i mapira labele u indekse klasa. DataLoader zatim služi da se podaci dovode u batch-evima, uz transformacije za trening i evaluaciju.

In [7]:
class ImageCsvDataset(Dataset):
    def __init__(self, df, path_col, label_col, label_to_idx=None, tfm=None):
        self.df = df.reset_index(drop=True)
        self.path_col = path_col
        self.label_col = label_col
        self.tfm = tfm

        labels = self.df[label_col].astype(str).tolist()
        if label_to_idx is None:
            uniq = sorted(list(set(labels)))
            self.label_to_idx = {u:i for i,u in enumerate(uniq)}
        else:
            self.label_to_idx = label_to_idx

        self.y = [self.label_to_idx[str(x)] for x in labels]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        p = str(self.df.iloc[i][self.path_col])
        y = self.y[i]
        im = Image.open(p).convert("RGB")
        if self.tfm is not None:
            im = self.tfm(im)
        return im, y

def make_image_loaders(ds, batch_size=32, num_workers=2, img_size=224):
    meta = load_meta(ds)
    df_tr = load_split(ds, "train")
    df_va = load_split(ds, "val")
    df_te = load_split(ds, "test")

    ycol = label_col(meta, df_tr)
    pcol = path_col(meta, df_tr)

    tfm_train = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
    ])
    tfm_eval = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])

    ds_tr = ImageCsvDataset(df_tr, pcol, ycol, label_to_idx=None, tfm=tfm_train)
    ds_va = ImageCsvDataset(df_va, pcol, ycol, label_to_idx=ds_tr.label_to_idx, tfm=tfm_eval)
    ds_te = ImageCsvDataset(df_te, pcol, ycol, label_to_idx=ds_tr.label_to_idx, tfm=tfm_eval)

    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    dl_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    idx_to_label = {v:k for k,v in ds_tr.label_to_idx.items()}
    return (dl_tr, dl_va, dl_te, ds_tr.label_to_idx, idx_to_label)

2.2.3 Model i petlja treninga

Učitava se pretrenirani ResNet18 i menja se poslednji sloj da odgovara broju klasa u konkretnom datasetu. Tokom treninga se prati loss, a na validaciji se bira najbolji model po macro F1 i na kraju se meri rezultat na testu.

In [8]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def build_resnet18(num_classes):
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        y = torch.tensor(y).to(device)
        logits = model(x)
        pred = torch.argmax(logits, dim=1)
        ys.extend(y.detach().cpu().numpy().tolist())
        ps.extend(pred.detach().cpu().numpy().tolist())
    acc = accuracy_score(ys, ps)
    f1m = f1_score(ys, ps, average="macro")
    return float(acc), float(f1m), ys, ps

def train_image_baseline(ds, epochs=3, lr=3e-4, batch_size=32, img_size=224):
    dl_tr, dl_va, dl_te, label_to_idx, idx_to_label = make_image_loaders(ds, batch_size=batch_size, img_size=img_size)
    num_classes = len(label_to_idx)

    model = build_resnet18(num_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    best = {"val_f1": -1, "state": None}

    hist = []
    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for x, y in dl_tr:
            x = x.to(device)
            y = torch.tensor(y).to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        tr_loss = float(np.mean(losses)) if losses else float("nan")
        va_acc, va_f1, _, _ = eval_epoch(model, dl_va)

        rec = {"epoch": ep, "train_loss": tr_loss, "val_acc": va_acc, "val_f1_macro": va_f1}
        hist.append(rec)

        if va_f1 > best["val_f1"]:
            best["val_f1"] = va_f1
            best["state"] = {k:v.cpu() for k,v in model.state_dict().items()}

        print(ds, rec)

    if best["state"] is not None:
        model.load_state_dict(best["state"])

    te_acc, te_f1, y_true, y_pred = eval_epoch(model, dl_te)

    return {
        "dataset": ds,
        "num_classes": num_classes,
        "label_to_idx": label_to_idx,
        "history": hist,
        "test": {"acc": te_acc, "f1_macro": te_f1, "y_true": y_true, "y_pred": y_pred}
    }

2.2.4 Pokretanje baseline-a za sve image datasete

Isti trening pipeline se izvršava redom za LC25000, SipakMed i RM1000, tako da dobiješ uporedive baseline metrike. Rezultati se skupljaju u tabelu radi lakšeg prebacivanja u Word i Excel.

In [9]:
image_results = []
for ds in ["lc25000", "sipakmed", "rm1000_lung_history"]:
    r = train_image_baseline(ds, epochs=3, lr=3e-4, batch_size=32, img_size=224)
    image_results.append(r)

pd.DataFrame([{
    "dataset": r["dataset"],
    "num_classes": r["num_classes"],
    "test_acc": r["test"]["acc"],
    "test_f1_macro": r["test"]["f1_macro"],
} for r in image_results])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 223MB/s]
/tmp/ipython-input-2836729689.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)
/tmp/ipython-input-2836729689.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)


lc25000 {'epoch': 1, 'train_loss': 0.005730564731907163, 'val_acc': 1.0, 'val_f1_macro': 1.0}


/tmp/ipython-input-2836729689.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)
/tmp/ipython-input-2836729689.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)


lc25000 {'epoch': 2, 'train_loss': 3.4868402568293756e-05, 'val_acc': 1.0, 'val_f1_macro': 1.0}


...

rm1000_lung_history {'epoch': 2, 'train_loss': 0.04186531818736526, 'val_acc': 0.9724444444444444, 'val_f1_macro': 0.9724547564262654}


/tmp/ipython-input-2836729689.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)
/tmp/ipython-input-2836729689.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)


rm1000_lung_history {'epoch': 3, 'train_loss': 0.040207862797101056, 'val_acc': 0.9995555555555555, 'val_f1_macro': 0.9995555553580245}


/tmp/ipython-input-2836729689.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).to(device)


,dataset,num_classes,test_acc,test_f1_macro
0,lc25000,2,1.000000,1.000000
1,sipakmed,5,0.957096,0.957348
2,rm1000_lung_history,3,0.999556,0.999556


2.3 Snimanje

2.3.1 Zajednicki helperi

In [10]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

RUNS = Path(RUNS)

run_id = time.strftime("%Y%m%d_%H%M%S")
out_root = RUNS / f"baselines_{run_id}"
out_root.mkdir(parents=True, exist_ok=True)

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def save_confmat_png(y_true, y_pred, labels, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, values_format="d")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

print("Saving to:", out_root)

Saving to: /content/drive/MyDrive/Personal/Diplomski/Runs/baselines_20260225_203344


2.3.2 Thyroid (tabular) LogReg i RandomForest

In [11]:
from sklearn.metrics import confusion_matrix

ds = "thyroid_recurrence"
ds_dir = out_root / ds
ds_dir.mkdir(parents=True, exist_ok=True)

meta = load_meta(ds)

config = {
    "dataset": ds,
    "task": "tabular",
    "seed": 42,
    "splits": {"train": "train.csv", "val": "val.csv", "test": "test.csv"},
    "meta_path": str((CURATED / ds / "meta.json").as_posix()),
    "timestamp": run_id
}
save_json(ds_dir / "config.json", config)

def flatten_report(rep):
    keep = {}
    for k, v in rep.items():
        if isinstance(v, dict):
            for kk, vv in v.items():
                if isinstance(vv, (int, float)):
                    keep[f"{k}.{kk}"] = float(vv)
        elif isinstance(v, (int, float)):
            keep[k] = float(v)
    return keep

tab_metrics = {
    "LogReg": {
        "val": res_lr["val"],
        "test": res_lr["test"],
        "test_report_flat": flatten_report(res_lr["report_test"]),
    },
    "RandomForest": {
        "val": res_rf["val"],
        "test": res_rf["test"],
        "test_report_flat": flatten_report(res_rf["report_test"]),
    }
}


X_test, y_test, _ = tabular_xy(ds, "test")
y_pred = rf.predict(X_test)

labels = sorted(list(pd.Series(y_test).astype(str).unique()))
label_to_idx = {l:i for i,l in enumerate(labels)}
y_true_idx = [label_to_idx[str(x)] for x in y_test]
y_pred_idx = [label_to_idx[str(x)] for x in y_pred]

save_confmat_png(y_true_idx, y_pred_idx, labels, f"{ds} RandomForest", ds_dir / "confusion_matrix.png")
print("Saved confusion matrix:", ds_dir / "confusion_matrix.png")

save_json(ds_dir / "metrics.json", {"meta": meta, "models": tab_metrics})

print("Saved:", ds_dir)

Saved confusion matrix: /content/drive/MyDrive/Personal/Diplomski/Runs/baselines_20260225_203344/thyroid_recurrence/confusion_matrix.png
Saved: /content/drive/MyDrive/Personal/Diplomski/Runs/baselines_20260225_203344/thyroid_recurrence


2.3.3 Image baselines (lc25000, sipakmed, rm1000_lung_history)

In [12]:
for r in image_results:
    ds = r["dataset"]
    ds_dir = out_root / ds
    ds_dir.mkdir(parents=True, exist_ok=True)

    meta = load_meta(ds)

    config = {
        "dataset": ds,
        "task": "image",
        "seed": 42,
        "model": "resnet18",
        "epochs": len(r["history"]),
        "timestamp": run_id,
        "meta_path": str((CURATED / ds / "meta.json").as_posix())
    }
    save_json(ds_dir / "config.json", config)

    save_json(ds_dir / "metrics.json", {
        "num_classes": r["num_classes"],
        "label_to_idx": r["label_to_idx"],
        "test": {"acc": r["test"]["acc"], "f1_macro": r["test"]["f1_macro"]},
        "history": r["history"]
    })

    pd.DataFrame(r["history"]).to_csv(ds_dir / "history.csv", index=False)

    idx_to_label = {int(k): v for k, v in {v:k for k,v in r["label_to_idx"].items()}.items()}  # safe cast
    labels = [idx_to_label[i] for i in range(r["num_classes"])]

    save_confmat_png(
        r["test"]["y_true"],
        r["test"]["y_pred"],
        labels,
        f"{ds} ResNet18",
        ds_dir / "confusion_matrix.png"
    )

    print("Saved:", ds_dir)

Saved: /content/drive/MyDrive/Personal/Diplomski/Runs/baselines_20260225_203344/lc25000
Saved: /content/drive/MyDrive/Personal/Diplomski/Runs/baselines_20260225_203344/sipakmed
Saved: /content/drive/MyDrive/Personal/Diplomski/Runs/baselines_20260225_203344/rm1000_lung_history


Zavrsni, rezime fajl

In [13]:
rows = []

rows.append({
    "dataset": "thyroid_recurrence",
    "model": "LogReg",
    "val_acc": res_lr["val"]["acc"],
    "val_f1_macro": res_lr["val"]["f1_macro"],
    "test_acc": res_lr["test"]["acc"],
    "test_f1_macro": res_lr["test"]["f1_macro"],
})

rows.append({
    "dataset": "thyroid_recurrence",
    "model": "RandomForest",
    "val_acc": res_rf["val"]["acc"],
    "val_f1_macro": res_rf["val"]["f1_macro"],
    "test_acc": res_rf["test"]["acc"],
    "test_f1_macro": res_rf["test"]["f1_macro"],
})

for r in image_results:
    rows.append({
        "dataset": r["dataset"],
        "model": "ResNet18",
        "val_acc": float(max([h["val_acc"] for h in r["history"]])),
        "val_f1_macro": float(max([h["val_f1_macro"] for h in r["history"]])),
        "test_acc": r["test"]["acc"],
        "test_f1_macro": r["test"]["f1_macro"],
    })

summary_df = pd.DataFrame(rows).sort_values(["dataset", "model"])
summary_df.to_csv(out_root / "summary.csv", index=False)
summary_df

,dataset,model,val_acc,val_f1_macro,test_acc,test_f1_macro
2,lc25000,ResNet18,1.000000,1.000000,1.000000,1.000000
4,rm1000_lung_history,ResNet18,0.999556,0.999556,0.999556,0.999556
3,sipakmed,ResNet18,0.945724,0.946147,0.957096,0.957348
0,thyroid_recurrence,LogReg,0.925926,0.903571,0.945455,0.932626
1,thyroid_recurrence,RandomForest,0.962963,0.953846,0.981818,0.977542
